In [33]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings

from tensorflow.python.distribute.distribute_lib import Strategy

warnings.filterwarnings("ignore")

In [34]:
# Set random seed for reproducibility
np.random.seed(42)

## Create Synthetic Data

In [35]:
print("=" * 70)
print("STEP 1: Creating Synthetic Patient Dataset")
print("=" * 70)

STEP 1: Creating Synthetic Patient Dataset


In [36]:
n_samples = 1000
data = {
    "age": np.random.randint(18, 80, size=n_samples),
    "blood_pressure": np.random.randint(90, 180, size=n_samples),
    "cholesterol": np.random.randint(150, 300, size=n_samples),
    "bmi": np.round(np.random.uniform(18.5, 40, size=n_samples), 1),
    "gender": np.random.choice(["Male", "Female"], size=n_samples),
    "smoking": np.random.choice(["Never", "Former", "Current"], size=n_samples),
    "exercise": np.random.choice(["Low", "Medium", "High"], size=n_samples),
    "family_history": np.random.choice(["Yes", "No"], size=n_samples)
}
df = pd.DataFrame(data)
df.head()

,age,blood_pressure,cholesterol,bmi,gender,smoking,exercise,family_history
0,56,100,155,29.2,Male,Never,Low,No
1,69,174,296,25.8,Female,Former,High,Yes
2,46,115,280,37.3,Female,Current,Low,Yes
3,32,152,150,30.2,Male,Never,High,No
4,60,178,207,34.8,Female,Current,Medium,Yes


In [37]:
# Introduce some missing values
missing_indices_bp = np.random.choice(a=n_samples, size=50, replace=False)
missing_indices_chol = np.random.choice(a=n_samples, size=50, replace=False)

df.loc[missing_indices_bp, "blood_pressure"] = np.nan
df.loc[missing_indices_chol, "cholesterol"] = np.nan

In [38]:
# Create a target variable (disease risk: 0=low, 1=High)

df["disease_risk"] = (
    (df['age'] > 55).astype(int) +
    (df['blood_pressure'].fillna(df['blood_pressure'].mean()) > 140).astype(int) +
    (df['cholesterol'].fillna(df['cholesterol'].mean()) > 240).astype(int) +
    (df['bmi'] > 30).astype(int) +
    (df['smoking'] == 'Current').astype(int) +
    (df['exercise'] == 'Low').astype(int) +
    (df['family_history'] == 'Yes').astype(int)
)
df["disease_risk"] = (df["disease_risk"] >= 3).astype(int)
df

,age,blood_pressure,cholesterol,bmi,gender,smoking,exercise,family_history,disease_risk
0,56,100.0,155.0,29.2,Male,Never,Low,No,0
1,69,174.0,296.0,25.8,Female,Former,High,Yes,1
2,46,115.0,280.0,37.3,Female,Current,Low,Yes,1
3,32,152.0,150.0,30.2,Male,Never,High,No,0
4,60,178.0,NaN,34.8,Female,Current,Medium,Yes,1
...,...,...,...,...,...,...,...,...,...
995,18,110.0,187.0,27.8,Female,Current,High,No,0
996,35,136.0,NaN,33.5,Female,Former,Low,No,0
997,49,169.0,287.0,28.3,Male,Former,High,Yes,1
998,64,148.0,248.0,20.5,Female,Current,Medium,No,1


In [39]:
# Shape
print("Shape", df.shape)

Shape (1000, 9)


In [40]:
# Missing Values
df.isnull().sum()

age                0
blood_pressure    50
cholesterol       50
bmi                0
gender             0
smoking            0
exercise           0
family_history     0
disease_risk       0
dtype: int64

In [41]:
# Target Distribution
df["disease_risk"].value_counts()

disease_risk
1    581
0    419
Name: count, dtype: int64

## Split the data

In [42]:
X = df.drop("disease_risk", axis=1)
y = df["disease_risk"]

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.75, random_state=42, stratify=y)

In [43]:
print("Training Data Shape", X_train.shape)
print("Testing Data Shape", X_test.shape)

Training Data Shape (750, 8)
Testing Data Shape (250, 8)


## Define Column Groups - Identifying Column Types

In [44]:
df.columns

Index(['age', 'blood_pressure', 'cholesterol', 'bmi', 'gender', 'smoking',
       'exercise', 'family_history', 'disease_risk'],
      dtype='object')

In [45]:
# Numerical Features that need imputation and scaling
num_features = ["age", "blood_pressure", "cholesterol", "bmi"]

# Categorical Features (nominal)
nominal_cat = ["gender", "smoking", "family_history"]

# Categorical Features (ordinal)
ordinal_cat = ["exercise"]
exercise_order = ["Low", "Medium", "High"]

In [46]:
print("Numerical Features", num_features)
print("Nominal Categorical Features", nominal_cat)
print("Ordinal Categorical Features", ordinal_cat)

Numerical Features ['age', 'blood_pressure', 'cholesterol', 'bmi']
Nominal Categorical Features ['gender', 'smoking', 'family_history']
Ordinal Categorical Features ['exercise']


## Prepare Preprocessing PIPELINES for each column type

In [ ]:
# Create a mini pipeline for numerical columns. Numerical Columns need imputation and scaling

In [47]:
# Pipeline for Numerical Features
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
])

In [48]:
# Pipeline for Nominal Categorical Features
nominal_cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"))
])

In [49]:
# Piepline for Ordinal Categorical Features
ordinal_cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(categories=[exercise_order], handle_unknown="use_encoded_value", unknown_value=-1))
])

In [50]:
print("\n✓ Numerical Pipeline: Impute (mean) → Scale")
print("✓ Nominal Categorical Pipeline: Impute (mode) → OneHotEncode")
print("✓ Ordinal Categorical Pipeline: Impute (mode) → OrdinalEncode")


✓ Numerical Pipeline: Impute (mean) → Scale
✓ Nominal Categorical Pipeline: Impute (mode) → OneHotEncode
✓ Ordinal Categorical Pipeline: Impute (mode) → OrdinalEncode


## Create All preprocessing with ColumnTransformer

In [51]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, num_features),
        ("cat_nom", nominal_cat_pipeline, nominal_cat),
        ("ord_cat", ordinal_cat_pipeline, ordinal_cat)
    ],
    remainder="drop"
)

## Create Complete Pipeline (Preprocessing + Model)

In [56]:
# Chains preprocessing with the final model
full_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
])

## Train the pipeline

In [53]:
# Numerical imputer learns mean from training bp, cholesterol
# Scaler learns mean/std from training numerical feature
# OneHotEncoder learns all categories from training data
# All transformers transform the training data
# Random Forest trains on the transformed features
full_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat_nom', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## Make Predictions

In [54]:
# What happens internally:
# Imputes missing values in the test set using TRAINING means
# Scales test data using TRAINING mean/std
# Encodes test categories using TRAINING encodings
# Random Forest predicts on transformed test data
y_pred = full_pipeline.predict(X_test)
y_pred

array([1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1,
       1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1,
       1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1,
       1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1,
       0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0,
       1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1,
       0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1,
       1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1,
       0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0,
       1, 1, 1, 0, 1, 1, 1, 1])

## Evaluate the model

In [55]:
accuracy = accuracy_score(y_test, y_pred)
print(f"\nAccuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Low Risk', 'High Risk']))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Accuracy: 0.8920

Classification Report:
              precision    recall  f1-score   support

    Low Risk       0.91      0.82      0.86       105
   High Risk       0.88      0.94      0.91       145

    accuracy                           0.89       250
   macro avg       0.90      0.88      0.89       250
weighted avg       0.89      0.89      0.89       250


Confusion Matrix:
[[ 86  19]
 [  8 137]]
